# Temporal Prerequisite-Aware Graph Knowledge Tracing (TPA-GKT)
## Paper 1 Full-Scale Benchmark: Multi-Seed Training, Ablations & Evaluation on Junyi Academy 2020

This notebook trains and benchmarks **TPA-GKT** against fair baselines (**DKT**, **SAKT**, **GKT**) and executes the complete 6-part ablation suite across seeds `[42, 100, 2026]` on a **Google Colab T4 GPU**.

### Pipeline Overview
1. **Environment Verification & Google Drive Mount**: Verifies GPU and establishes cloud persistence.
2. **Dependencies & Repo Setup**: Clones branch `research/tpa-gkt-junyi` and installs PyTorch Geometric.
3. **Full-Scale Preprocessing & Graph Construction**: Ingests Junyi raw CSVs (`max_students=5000`, `top_n_concepts=150`) and builds $A_{\text{hier}}$, $A_{\text{trans}}$, and $A_{\text{combined}}$.
4. **Multi-Seed Training & Ablation Execution**: Runs 10 model configs $\times$ 3 seeds (30 runs total) with strict validation early stopping.
5. **Publication-Grade Figures & Cloud Sync**: Generates 300 DPI figures (`roc_comparison.png`, `retention_by_delay_bucket.png`, `calibration_reliability.png`) and syncs all checkpoints, tables, and logs to Google Drive.

### Step 1: Environment Verification & Google Drive Mount

In [ ]:
# Mount Google Drive for zero-loss checkpoint and dataset persistence
import os

import torch
from google.colab import drive

drive.mount("/content/drive")

# Verify CUDA GPU availability
!nvidia-smi

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n[OK] Active Device: {device}")
if torch.cuda.is_available():
    print(f"[OK] GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"[OK] Device Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print(
        "[WARN] Running on CPU. Please switch Colab Runtime to T4 GPU (Runtime -> Change runtime type -> T4 GPU)."
    )

# Establish Google Drive storage paths
DRIVE_RAW_DIR = "/content/drive/MyDrive/cognitwin_data/raw"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/cognitwin_outputs"

os.makedirs(DRIVE_RAW_DIR, exist_ok=True)
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
print(f"[OK] Persistent Output Directory: {DRIVE_OUTPUT_DIR}")

### Step 2: Repository Setup & Dependency Installation

In [ ]:
# Clone or pull latest research branch
REPO_URL = "https://github.com/AKYADAV0405/cognitwin.git"
BRANCH = "research/tpa-gkt-junyi"
REPO_DIR = "/content/cognitwin"

if os.path.exists(REPO_DIR):
    %cd $REPO_DIR
    !git checkout $BRANCH
    !git pull
else:
    # If cloning from private or public git repo
    !git clone -b $BRANCH $REPO_URL $REPO_DIR || echo "If using local folder, copy project to /content/cognitwin"
    %cd $REPO_DIR

# Install PyTorch Geometric and required packages compatible with Colab CUDA
!pip install -q torch-geometric scikit-learn pyyaml pandas pyarrow matplotlib ruff pytest

# Verify installation
import sklearn
import torch_geometric

print(f"[OK] PyTorch Geometric Version: {torch_geometric.__version__}")
print(f"[OK] scikit-learn Version: {sklearn.__version__}")

### Step 3: Full-Scale Preprocessing & Knowledge Graph Construction
*Ingests Junyi Academy 2020 (`5,000` students, `150` concepts, 70/15/15 student-disjoint split).* 

In [ ]:
# Fast restore check from Google Drive if processed data already exists
drive_proc = "/content/drive/MyDrive/cognitwin_data/processed"
if os.path.exists(f"{drive_proc}/train.parquet") and os.path.exists(f"{drive_proc}/A_combined.pt"):
    print(
        "[OK] Found existing preprocessed data in Google Drive. Fast-restoring to data/processed/..."
    )
    !mkdir -p data/processed
    !cp -r /content/drive/MyDrive/cognitwin_data/processed/* data/processed/
    print("[OK] Preprocessed data successfully restored from Google Drive!")
else:
    # Link raw CSVs from Google Drive to local repository data/raw/
    !mkdir -p data/raw
    !cp /content/drive/MyDrive/cognitwin_data/raw/*.csv data/raw/ || echo "Checking local data/raw/..."

    # Verify raw CSV existence
    assert os.path.exists("data/raw/Info_Content.csv"), "Info_Content.csv missing in data/raw/"
    assert os.path.exists("data/raw/Log_Problem.csv"), "Log_Problem.csv missing in data/raw/"

    print("[OK] Starting Zero-Leakage Preprocessing (5,000 students, Top 150 concepts)...")
    !python -u services/gnn_kt/src/preprocess.py --config services/gnn_kt/config.yaml
    if _exit_code != 0:
        raise RuntimeError(f"preprocess.py failed with exit code {_exit_code}!")

    print("\n[OK] Building Dual Knowledge Graph (A_hier, A_trans, A_combined)...")
    !python -u services/gnn_kt/src/graph_builder.py --config services/gnn_kt/config.yaml
    if _exit_code != 0:
        raise RuntimeError(f"graph_builder.py failed with exit code {_exit_code}!")

    # Backup preprocessed parquet datasets to Drive for immediate re-use
    !mkdir -p /content/drive/MyDrive/cognitwin_data/processed
    !cp -r data/processed/* /content/drive/MyDrive/cognitwin_data/processed/
    print("[OK] Preprocessed data safely backed up to Google Drive.")

### Step 4: Automated Multi-Seed Benchmark & Ablation Suite
*Runs all 4 models (`DKT`, `SAKT`, `GKT`, `TPA-GKT`) and all 6 ablations across seeds `[42, 100, 2026]` on T4 GPU.*

In [ ]:
# Pre-flight check: ensure preprocessed data exists before launching 30 runs
assert os.path.exists("data/processed/test.parquet"), (
    "[ERROR] data/processed/test.parquet not found! Please run Step 3 (Preprocessing & Knowledge Graph) first."
)

# Optional: set CUDA_LAUNCH_BLOCKING=1 if deep stack trace debugging is needed
# import os; os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

# Freeze experiment configuration snapshot
!mkdir -p services/gnn_kt/outputs/results
!cp services/gnn_kt/config.yaml services/gnn_kt/outputs/experiment_config_snapshot.yaml

# Launch Multi-Seed Benchmark Suite with real-time live output streaming and strict error halting
print("[INFO] Launching Multi-Seed Benchmark Suite across seeds [42, 100, 2026]...")
!python -u services/gnn_kt/src/run_ablations.py --seeds 42 100 2026
if _exit_code != 0:
    raise RuntimeError(f"run_ablations.py failed with exit code {_exit_code}! Aborting pipeline.")

# Sync all checkpoints, tables, and runs immediately to Drive
!mkdir -p /content/drive/MyDrive/cognitwin_outputs/results
!mkdir -p /content/drive/MyDrive/cognitwin_outputs/checkpoints
!cp -r services/gnn_kt/outputs/results/* /content/drive/MyDrive/cognitwin_outputs/results/
!cp -r services/gnn_kt/outputs/checkpoints/* /content/drive/MyDrive/cognitwin_outputs/checkpoints/
!cp services/gnn_kt/outputs/experiment_config_snapshot.yaml /content/drive/MyDrive/cognitwin_outputs/

print("[OK] Benchmarking complete. All 30 models and results safely backed up to Google Drive.")

### Step 5: Publication Figures & Results Tables Inspection

In [ ]:
import pandas as pd
from IPython.display import Image, display

# Generate 300 DPI publication figures
!python services/gnn_kt/src/plot_results.py --results-dir services/gnn_kt/outputs/results

# Sync figures to Google Drive
!cp -r services/gnn_kt/outputs/results/figures /content/drive/MyDrive/cognitwin_outputs/

print("=" * 80)
print("TABLE 3: BASELINES VS. TPA-GKT (OURS) ON JUNYI ACADEMY 2020 (MEAN ± STD)")
print("=" * 80)
df_table3 = pd.read_csv("services/gnn_kt/outputs/results/table3_results.csv")
display(df_table3)

print("\n" + "=" * 80)
print("ABLATION SUITE (ALL 6 ARCHITECTURAL VARIANTS)")
print("=" * 80)
df_ablations = pd.read_csv("services/gnn_kt/outputs/results/ablations.csv")
display(df_ablations)

print("\n" + "=" * 80)
print("DELAYED RETENTION EVALUATION ACROSS TIME INTERVALS")
print("=" * 80)
df_buckets = pd.read_csv("services/gnn_kt/outputs/results/retention_buckets.csv")
display(df_buckets)

print("\n" + "=" * 80)
print("PUBLICATION FIGURES PREVIEW")
print("=" * 80)
for fig_name in [
    "roc_comparison.png",
    "retention_by_delay_bucket.png",
    "calibration_reliability.png",
]:
    fig_path = f"services/gnn_kt/outputs/results/figures/{fig_name}"
    if os.path.exists(fig_path):
        print(f"\nDisplaying: {fig_name}")
        display(Image(filename=fig_path))

print(
    "\n[SUCCESS] Entire TPA-GKT Paper 1 GPU Run has completed and is fully secured in Google Drive!"
)

### Step 6: Final Experiment Validation Audit (Advisor Checks 1–5)
*Directly audits saved 5,000-student × 3-seed checkpoints and test parquets:
  1. Seed-level results verification & dataset fingerprint assertion
  2. Graph embedding geometry & sparse concept stratification
  3. Multi-step independent concept mastery validation (k >= 3)
  4. Forensic delayed-retention label & zero-leakage audit
  5. Student-clustered paired bootstrap (B=1,000) statistical significance*

In [ ]:
import os

import pandas as pd
from IPython.display import Markdown, display

# 1. Fast restore check from Google Drive if checkpoints/outputs are not in local runtime
drive_out = "/content/drive/MyDrive/cognitwin_outputs"
if os.path.exists(f"{drive_out}/checkpoints") and not os.path.exists(
    "services/gnn_kt/outputs/checkpoints/tpa_gkt_full_seed42_best.pt"
):
    print("[OK] Restoring saved checkpoints and results from Google Drive...")
    !mkdir -p services/gnn_kt/outputs/checkpoints services/gnn_kt/outputs/results services/gnn_kt/outputs/runs
    !cp -r /content/drive/MyDrive/cognitwin_outputs/checkpoints/* services/gnn_kt/outputs/checkpoints/ || true
    !cp -r /content/drive/MyDrive/cognitwin_outputs/results/* services/gnn_kt/outputs/results/ || true
    !cp -r /content/drive/MyDrive/cognitwin_outputs/runs/* services/gnn_kt/outputs/runs/ || true
    print("[OK] Checkpoints and runs successfully restored from Google Drive!")

# Pre-flight assertions
assert os.path.exists("data/processed/test.parquet"), "data/processed/test.parquet missing!"

# 2. Run Comprehensive Final Validation Audit
print("[INFO] Launching Final Experiment Validation Audit across all 5 checks...")
!python -u services/gnn_kt/src/final_validation_audit.py \
    --results-dir services/gnn_kt/outputs/results \
    --runs-dir services/gnn_kt/outputs/runs \
    --checkpoints-dir services/gnn_kt/outputs/checkpoints \
    --data-dir data/processed \
    --n-bootstrap 1000

if _exit_code != 0:
    raise RuntimeError(f"final_validation_audit.py failed with exit code {_exit_code}!")

# 3. Display Key Validation Tables Inline
print("\n" + "=" * 80)
print("CHECK 1: PER-SEED BENCHMARK RESULTS")
print("=" * 80)
df_seeds = pd.read_csv("services/gnn_kt/outputs/results/per_seed_results.csv")
display(df_seeds.head(15))

print("\n" + "=" * 80)
print("CHECK 5: PAIRED STATISTICAL SIGNIFICANCE (STUDENT-CLUSTERED BOOTSTRAP B=1,000)")
print("=" * 80)
df_stats = pd.read_csv("services/gnn_kt/outputs/results/statistical_tests.csv")
display(df_stats)

# 4. Display Executive Summary from Generated Markdown
summary_md = "services/gnn_kt/outputs/results/final_experiment_validation.md"
if os.path.exists(summary_md):
    with open(summary_md, encoding="utf-8") as f:
        print("\n" + "=" * 80)
        print("EXECUTIVE AUDIT SUMMARY")
        print("=" * 80)
        display(Markdown(f.read()))

# 5. Sync all new audit files back to Google Drive
!mkdir -p /content/drive/MyDrive/cognitwin_outputs/results
!cp services/gnn_kt/outputs/results/per_seed_results.csv /content/drive/MyDrive/cognitwin_outputs/results/
!cp services/gnn_kt/outputs/results/statistical_tests.csv /content/drive/MyDrive/cognitwin_outputs/results/
!cp services/gnn_kt/outputs/results/graph_ablation_audit.json /content/drive/MyDrive/cognitwin_outputs/results/
!cp services/gnn_kt/outputs/results/mastery_independent_eval.json /content/drive/MyDrive/cognitwin_outputs/results/
!cp services/gnn_kt/outputs/results/retention_label_audit.json /content/drive/MyDrive/cognitwin_outputs/results/
!cp services/gnn_kt/outputs/results/final_experiment_validation.md /content/drive/MyDrive/cognitwin_outputs/results/

print(
    "\n[SUCCESS] Final Experiment Validation Audit complete. All audit tables and reports backed up to Google Drive!"
)